# Crypto Sentiment Analysis — Exploratory Notebook

End-to-end: load data → EDA → preprocess → train → evaluate → demo

In [ ]:
import sys, os
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from preprocess import clean_text, clean_batch
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (11, 5)
print('Ready')

## 1. Load Data

In [ ]:
news   = pd.read_csv('../data/crypto_news_mendeley.csv')
tweets = pd.read_csv('../data/twitter_sample.csv')
samples = pd.read_csv('../data/news_sample.csv')
samples['text'] = samples['title'] + ' ' + samples['description'].fillna('')
combined = pd.concat([
    news[['text','label']].assign(source='mendeley'),
    tweets[['text','label']].assign(source='twitter'),
    samples[['text','label']].assign(source='news'),
], ignore_index=True)
combined['label'] = combined['label'].str.lower().str.strip()
combined = combined[combined['label'].isin(['positive','negative','neutral'])]
print(f'Total: {len(combined)}')
combined.head()

## 2. Class Distribution

In [ ]:
colors = {'positive':'#22d98a','negative':'#ff4f6d','neutral':'#7b91ff'}
fig, axes = plt.subplots(1,2,figsize=(14,5))
counts = combined['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=[colors[l] for l in counts.index])
axes[0].set_title('Overall distribution')
combined.groupby(['source','label']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[1], color=[colors.get(c,'grey') for c in combined['label'].unique()])
axes[1].set_title('By source'); axes[1].tick_params(axis='x',rotation=0)
plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing Demo

In [ ]:
examples = [
    '$BTC to the moon!! #Bitcoin #HODL best investment ever',
    'Crypto REKT after hack. FUD spreading fast http://t.co/xyz',
    'Just another boring day in crypto. Nothing happening.',
]
for e in examples:
    print(f'IN : {e}')
    print(f'OUT: {clean_text(e)}\n')
combined['cleaned'] = clean_batch(combined['text'].tolist())
print('Preprocessing complete')
combined[['text','cleaned','label']].head()

## 4. Top Keywords Per Class

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,6))
for ax, label in zip(axes, ['positive','negative','neutral']):
    tokens = ' '.join(combined[combined['label']==label]['cleaned']).split()
    tokens = [t for t in tokens if len(t) > 3]
    top = Counter(tokens).most_common(15)
    words, cnts = zip(*top) if top else ([],[])
    ax.barh(list(words)[::-1], list(cnts)[::-1], color=colors[label])
    ax.set_title(f'{label.capitalize()} keywords')
plt.tight_layout()
plt.savefig('../outputs/plots/top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Train & Evaluate

In [ ]:
from train import train
results = train(combined)
m = results['metrics']
print(f"Accuracy : {m['accuracy']}")
print(f"CV Acc   : {m['cv_accuracy_mean']} ± {m['cv_accuracy_std']}")

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np
cr = m['classification_report']
labels = m['classes']
fig, ax = plt.subplots(figsize=(7,5))
from sklearn.preprocessing import LabelEncoder
le = results['label_encoder']
from sklearn.model_selection import train_test_split
from preprocess import clean_batch
X_clean = clean_batch(combined['text'].tolist())
from sklearn.feature_extraction.text import TfidfVectorizer
# quick re-vectorise for display
vec = results['vectorizer']
X = vec.transform(X_clean)
y = le.transform(combined['label'])
_, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred = results['clf'].predict(X_te)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('../outputs/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Prediction Demo

In [ ]:
from predict import predict_one
demos = [
    'Bitcoin just hit a new all-time high! Incredible bull run this week!',
    'Lost everything in the crypto crash. This market is an absolute disaster.',
    'BTC price remains stable today. No major moves expected.',
    'DeFi yields are insane right now! Staking returns look very attractive.',
    'Another exchange hack reported. Security in crypto remains a huge problem.',
]
for d in demos:
    r = predict_one(d)
    bar = '█' * int(r['score']*20)
    print(f"{r['sentiment'].upper():8} [{bar:<20}] {r['score']:.2f}  {d[:60]}")